# Treinamento ResNet + Lstm Multimodal

Será implementada uma nova loss baseada em BCE e Margin Ranking Loss.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 1. Setup

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import sys
import datetime
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

import optuna
from transformers import AutoModel # transformers==4.44.0
import einops
import timm
import torchaudio # torchaudio==2.5.1

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

from models.nn.resnetlstm import ResNetLSTM
from models.nn.resnetlstm_multimodal import ResNetLSTM_MultiModal
#from preprocessing.loader_multimodal_pair import train_video_transform, train_mel_transform, TARGET_SHAPE, build_multimodal_dataloader
from preprocessing.loader_multimodal_frac2 import build_multimodal_dataloader, train_video_transform, TARGET_SHAPE, train_mel_transform
from preprocessing.balanced_dataset import balanced_df

/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 435
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch: 2.5.1+cu121
Torchvision: 0.20.1+cu121
CUDA disponível: True
CUDA: 12.1
GPU: NVIDIA RTX A4500
Device: cuda


## 2. Configuração

In [5]:
# paths
LABELS_ALL   = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_combinadas_wg.csv"
LABELS_DIR   = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used"
CKPT_DIR     = "/mnt/storage_C4/gaussian_football/models/checkpoints"
RUNS_DIR     = "/mnt/storage_C4/gaussian_football/runs_multimodal_daug_more_games"  # logs do TensorBoard

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# treino 
EPOCHS        = 100        # teto por causa do early stopping decide
PATIENCE      = 20         # épocas sem melhora antes de parar
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
MONITOR       = "loss"      # métrica de checkpoint/early-stopping: "ccc" | "mae" | "loss"

# dataloaders (mudei pra 1 pra rodar a resnet 50 c finetune)
BATCH_SIZE = 1
BATCH_SIZE_RES50 = 1

# modelo padrão
MODEL_DEFAULTS = dict(
    frame_step=1,
    hidden_size=256,
    num_layers=1,
    use_dropout=False,
    dropout_p=0.3,
)

## 3. Balanceando os Dados

Gera os CSVs balanceados (50% highlight / 50% não-highlight por partida,
com `threshold=0.5` no `arousal_score`). Só recalcula se os arquivos não existirem
(use `FORCE_REBUILD = True` para refazer).

In [6]:

FORCE_REBUILD = True

paths_balanced = {
    "train": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_train_wg.csv"),
    "valid": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_valid_wg.csv"),
    "test":  os.path.join(LABELS_DIR, "balanced_todas_as_ligas_test_wg.csv"),
}

if FORCE_REBUILD or not all(os.path.exists(p) for p in paths_balanced.values()):
    all_data = pd.read_csv(LABELS_ALL)
    for split, out_path in paths_balanced.items():
        subset = all_data[all_data["split"] == split]
        balanced = balanced_df(subset, "game_id", threshold=0.5, random_state=SEED)
        balanced.to_csv(out_path, index=False)
        print(f"{split}: {len(balanced)} clipes -> {out_path}")
else:
    print("CSVs balanceados já existem. (FORCE_REBUILD=True para refazer.)")

train: 16912 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_train_wg.csv
valid: 6152 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_valid_wg.csv
test: 5730 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_test_wg.csv


## 4. DataLoaders Multimodais

In [7]:
TRAIN_PATH = f"{LABELS_DIR}/todas_as_ligas_train_wg.csv"
VAL_PATH   = f"{LABELS_DIR}/todas_as_ligas_valid_wg.csv"
TEST_PATH   = f"{LABELS_DIR}/todas_as_ligas_test_wg.csv"

In [8]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df['type'].value_counts()

type
Background    56396
shot          47936
goal           8460
Name: count, dtype: int64

Filtrando os datasets apenas por `goal` (high score) e `Background` (low score).

In [9]:
eventos_usados = ['goal', 'Background']
dir_background_goal = '/mnt/storage_C4/gaussian_football/data/datasets_goal_background'

# train
train_filtrado = train_df[train_df['type'].isin(eventos_usados)]
print('train:\n', train_filtrado['type'].value_counts())
train_filtrado.to_csv(f'{dir_background_goal}/train.csv', index=False)

# val
val_filtrado = val_df[val_df['type'].isin(eventos_usados)]
print('\n val:\n', val_filtrado['type'].value_counts())
val_filtrado.to_csv(f'{dir_background_goal}/val.csv', index=False)

# test
test_filtrado = test_df[test_df['type'].isin(eventos_usados)]
print('\n test:\n', test_filtrado['type'].value_counts())
test_filtrado.to_csv(f'{dir_background_goal}/test.csv', index=False)

train:
 type
Background    56396
goal           8460
Name: count, dtype: int64

 val:
 type
Background    17917
goal           3076
Name: count, dtype: int64

 test:
 type
Background    18697
goal           2865
Name: count, dtype: int64


In [10]:
# novos caminhos para os datasets que vamos usar:
TRAIN_PATH = '/mnt/storage_C4/gaussian_football/data/datasets_goal_background/train.csv'
VAL_PATH = '/mnt/storage_C4/gaussian_football/data/datasets_goal_background/val.csv'
TEST_PATH = '/mnt/storage_C4/gaussian_football/data/datasets_goal_background/test.csv'

## Sessão de Cuidados com Clipes Corrompidos

In [11]:
import av
import re
import pandas as pd
from tqdm.auto import tqdm

av.logging.set_level(av.logging.PANIC)

def check_video(path):
    '''Tenta abrir e decodificar 1 frame do vídeo pra confirmar que ele não está corrompido.'''
    try:
        container = av.open(path)
        stream = container.streams.video[0]
        next(container.decode(stream))
        container.close()
        return True
    except Exception:
        return False

def find_corrupted(df, col="clip_path"):
    '''Roda check_video em todos os paths de uma coluna, mostrando contagem em tempo real.'''
    bad = []
    pbar = tqdm(df[col], desc="checando vídeos")
    for p in pbar:
        if not check_video(p):
            bad.append(p)
        pbar.set_postfix(corrompidos=len(bad))
    return bad

def report_by_league_and_game(df, bad_paths, split_name):
    '''Mostra a distribuição dos corrompidos por liga (extraída do game_id) e por jogo.'''
    if not bad_paths:
        print(f"=== {split_name}: nenhum corrompido ===\n")
        return

    bad_df = df[df['clip_path'].isin(bad_paths)].copy()
    bad_df['league'] = bad_df['game_id'].str.split('_').str[0]

    print(f"=== {split_name} ===")
    print("corrompidos por liga:")
    print(bad_df['league'].value_counts())
    print()
    print(bad_df['game_id'].nunique(), "jogos distintos afetados")
    print(bad_df.groupby('game_id').size().sort_values(ascending=False).head(10))
    print()

def filter_by_window(df, bad_paths, key_col="window_id"):
    '''Remove o grupo de pareamento (window_id) inteiro se qualquer clipe dele estiver corrompido,
    evitando deixar um clipe low/high órfão (sem par) pro dataloader com pair=True.'''
    bad_windows = df.loc[df['clip_path'].isin(bad_paths), key_col].unique()
    return df[~df[key_col].isin(bad_windows)]

In [54]:
# identifica corrompidos nos três splits 
bad_train = find_corrupted(train_filtrado)
bad_val = find_corrupted(val_filtrado)
bad_test = find_corrupted(test_filtrado)

print(len(bad_train), "train corrompidos")
print(len(bad_val), "val corrompidos")
print(len(bad_test), "test corrompidos")

checando vídeos: 100%|██████████| 21562/21562 [01:06<00:00, 325.59it/s, corrompidos=2795] 

392 train corrompidos
1531 val corrompidos
2795 test corrompidos


In [58]:
# relatório por liga e jogo
report_by_league_and_game(train_filtrado, bad_train, "train")
report_by_league_and_game(val_filtrado, bad_val, "val")
report_by_league_and_game(test_filtrado, bad_test, "test")

=== train ===
corrompidos por liga:
league
ligue-1    392
Name: count, dtype: int64

2 jogos distintos afetados
game_id
ligue-1_2016-2017_2017-05-20-22-00-paris-sg-1-1-caen      205
ligue-1_2016-2017_2017-05-06-18-00-paris-sg-5-0-bastia    187
dtype: int64

=== val ===
corrompidos por liga:
league
ligue-1    1531
Name: count, dtype: int64

8 jogos distintos afetados
game_id
ligue-1_2016-2017_2016-08-21-21-45-paris-sg-3-0-metz           328
ligue-1_2016-2017_2017-03-04-19-00-paris-sg-1-0-nancy          232
ligue-1_2016-2017_2017-04-09-22-00-paris-sg-4-0-guingamp       212
ligue-1_2016-2017_2017-05-14-22-00-st-etienne-0-5-paris-sg     212
ligue-1_2016-2017_2016-09-20-22-00-paris-sg-3-0-dijon          164
ligue-1_2016-2017_2016-09-09-21-45-paris-sg-1-1-st-etienne     154
ligue-1_2016-2017_2016-12-03-19-00-montpellier-3-0-paris-sg    146
ligue-1_2016-2017_2016-10-23-21-45-paris-sg-0-0-marseille       83
dtype: int64

=== test ===
corrompidos por liga:
league
laliga                   2146
u

In [59]:
# salva lista dos corrompidos
all_bad = bad_train + bad_val + bad_test
pd.Series(all_bad).to_csv(f'{dir_background_goal}/videos_corrompidos.csv', index=False)

In [60]:
# filtra por window_id inteiro pra não deixar um clipe só
train_clean = filter_by_window(train_filtrado, bad_train)
val_clean = filter_by_window(val_filtrado, bad_val)
test_clean = filter_by_window(test_filtrado, bad_test)

train_clean.to_csv(f'{dir_background_goal}/train_filtered.csv', index=False)
val_clean.to_csv(f'{dir_background_goal}/val_filtered.csv', index=False)
test_clean.to_csv(f'{dir_background_goal}/test_filtered.csv', index=False)

print(len(train_filtrado) - len(train_clean), "linhas removidas do train (incluindo pares órfãos)")
print(len(val_filtrado) - len(val_clean), "linhas removidas do val (incluindo pares órfãos)")
print(len(test_filtrado) - len(test_clean), "linhas removidas do test (incluindo pares órfãos)")

392 linhas removidas do train (incluindo pares órfãos)
1531 linhas removidas do val (incluindo pares órfãos)
2815 linhas removidas do test (incluindo pares órfãos)


## Daqui pra frente tá de boa

In [11]:
# atualiza os paths que o dataloader vai ler
TRAIN_PATH = f'{dir_background_goal}/train_filtered.csv'
VAL_PATH = f'{dir_background_goal}/val_filtered.csv'
TEST_PATH = f'{dir_background_goal}/test_filtered.csv'

print("\nTRAIN_PATH, VAL_PATH, TEST_PATH atualizados")


TRAIN_PATH, VAL_PATH, TEST_PATH atualizados


In [12]:
'''
train_loader_bs2 = build_multimodal_dataloader(
    csv_path=TRAIN_PATH,
    pair=True, # pareamento dos dados low e high
    split="train",
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
    video_transform=train_video_transform,
    mel_transform=train_mel_transform,
)

valid_loader_bs2 = build_multimodal_dataloader(
    csv_path=VAL_PATH,
    pair=True, # pareamento dos dados para loss marging ranking 
    split='valid',
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

train_loader_bs1 = build_multimodal_dataloader(
    csv_path=TRAIN_PATH,
    pair=True, # pareamento dos dados low e high
    split="train",
    batch_size=BATCH_SIZE_RES50,
    shuffle=True,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
    video_transform=train_video_transform,
    mel_transform=train_mel_transform,
)

valid_loader_bs1 = build_multimodal_dataloader(
    csv_path=VAL_PATH,
    pair=True, # pareamento dos dados para loss marging ranking 
    split='valid',
    batch_size=BATCH_SIZE_RES50,
    shuffle=False,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

test_loader_bs2 = build_multimodal_dataloader(
    csv_path=TEST_PATH,
    pair=False,
    split='test',
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)
'''

'\ntrain_loader_bs2 = build_multimodal_dataloader(\n    csv_path=TRAIN_PATH,\n    pair=True, # pareamento dos dados low e high\n    split="train",\n    batch_size=BATCH_SIZE,\n    shuffle=True,\n    num_workers=4,\n    is_grayscale=True,\n    pin_memory=True,\n    video_transform=train_video_transform,\n    mel_transform=train_mel_transform,\n)\n\nvalid_loader_bs2 = build_multimodal_dataloader(\n    csv_path=VAL_PATH,\n    pair=True, # pareamento dos dados para loss marging ranking \n    split=\'valid\',\n    batch_size=BATCH_SIZE,\n    shuffle=False,\n    num_workers=4,\n    is_grayscale=True,\n    pin_memory=True,\n)\n\ntrain_loader_bs1 = build_multimodal_dataloader(\n    csv_path=TRAIN_PATH,\n    pair=True, # pareamento dos dados low e high\n    split="train",\n    batch_size=BATCH_SIZE_RES50,\n    shuffle=True,\n    num_workers=4,\n    is_grayscale=True,\n    pin_memory=True,\n    video_transform=train_video_transform,\n    mel_transform=train_mel_transform,\n)\n\nvalid_loader_bs1 

## 5. Métricas

#### CCC

O **CCC** (Concordance Correlation Coefficient) mede concordância entre predição e alvo,
punindo tanto erro de correlação quanto deslocamento de escala/média, por isso é sensível
ao *variance collapse* (modelo prevendo sempre perto da média).

$$\rho_c = \dfrac{2\,\rho\,\sigma_x\,\sigma_y}{\sigma_x^2 + \sigma_y^2 + (\mu_x - \mu_y)^2}$$

In [13]:
def calculate_ccc(y_true, y_pred):
    '''Concordance Correlation Coefficient sobre dois arrays 1D.'''
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true,  var_pred  = np.var(y_true),  np.var(y_pred)
    covariance = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    denominator = var_true + var_pred + (mean_true - mean_pred) ** 2
    return 0.0 if denominator == 0 else (2 * covariance) / denominator

### Perda Combinada

A função de perda usada irá combinar a BCE e a Margin Ranking Loss:

$$
Loss_{Total} = BCE_{Loss} + \lambda \cdot \text{Margin Ranking Loss}
$$

A Binary Cross Entropy vai penalizar diretamente desvios grandes na predição dos valores entre 0 e 1.

$$
BCE = - \frac{1}{N}\sum_{i=1}^{N}[y_i \cdot \log(\hat{y_i}) + (1-y_i)\cdot \log (1-\hat{y_i})]
$$

Enquanto o Margin Ranking Loss irá penalizar se o modelo atribuir baixa pontuação para amostras highlight e alta pontuação para amostras que não são highlights.

$$
Loss = \max (0, -Y \times (\hat{y}_{alto}- \hat{y}_{baixo}) + \text{margem})
$$

A margem utilizada será adaptativa, ou seja, para cada par de amostras a margem será calculada pela diferença das labels reais.

In [14]:
# CÓDIGO MORTO

"""def adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale=1.0):
    margin = margin_scale * (label_high - label_low)
    return torch.relu(margin - (pred_high - pred_low))

bce = nn.BCEWithLogitsLoss()

def combined_loss(label_low, label_high, pred_low, pred_high, alpha=1.0, margin_scale=1.0):
    loss_bce_low = bce(pred_low, label_low)
    loss_bce_high = bce(pred_high, label_high)
    loss_bce = 0.5 * (loss_bce_low + loss_bce_high)

    loss_rank = adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale,)

    return loss_bce + alpha * loss_rank"""

'def adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale=1.0):\n    margin = margin_scale * (label_high - label_low)\n    return torch.relu(margin - (pred_high - pred_low))\n\nbce = nn.BCEWithLogitsLoss()\n\ndef combined_loss(label_low, label_high, pred_low, pred_high, alpha=1.0, margin_scale=1.0):\n    loss_bce_low = bce(pred_low, label_low)\n    loss_bce_high = bce(pred_high, label_high)\n    loss_bce = 0.5 * (loss_bce_low + loss_bce_high)\n\n    loss_rank = adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale,)\n\n    return loss_bce + alpha * loss_rank'

In [15]:
class CombinedLoss(nn.Module):

    def __init__(self, alpha=1.0, margin_scale=1.0):
        super().__init__()

        self.alpha = alpha
        self.margin_scale = margin_scale
        self.bce = nn.BCEWithLogitsLoss()

    def forward(
        self,
        low_label,
        high_label,
        low_pred,
        high_pred,
        return_components=False,
    ):

        bce = (
            self.bce(low_pred, low_label)
            + self.bce(high_pred, high_label)
        ) / 2

        margin = self.margin_scale * (
            high_label - low_label
        )

        rank = torch.relu(
            margin - (high_pred - low_pred)
        ).mean()

        loss = bce + self.alpha * rank

        if return_components:
            return loss, bce, rank

        return loss

## 7. Treino

Função de treino unificada.

In [16]:
MONITOR_MODE = {"ccc": "max", "mae": "min", "loss": "min"}

def _apply_freeze(model, freeze_backbone):
    '''Liga/desliga o gradiente da ResNet. conv1 (índice 0) fica sempre treinável.'''
    for name, param in model.cnn.named_parameters():
        param.requires_grad = name.startswith("0.") or (not freeze_backbone)

def _set_backbone_eval(model):
    '''Congela estatísticas de BatchNorm na ResNet. Conv1 fica em train.'''
    for i, module in enumerate(model.cnn):
        module.train() if i == 0 else module.eval()

def binary_accuracy(y_true, y_pred, threshold=0.5):
    return float(np.mean((y_pred > threshold) == (y_true > threshold)))

def _is_better(current, best, mode):
    return current > best if mode == "max" else current < best

def _log_pred_scatter(writer, y_true, y_pred, tag, step):
    '''Scatter pred x target — faixa horizontal achatada = variance collapse.'''
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(y_true, y_pred, s=8, alpha=0.4)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "--", color="gray", linewidth=1)  # diagonal ideal
    ax.set_xlabel("target"); ax.set_ylabel("pred"); ax.set_title("pred vs target")
    writer.add_figure(tag, fig, step)
    plt.close(fig)


def train_model(
    model, train_loader, val_loader, *,
    criterion, run_name, optimizer, scheduler,
    backbone_name, loss_name,
    freeze_mode="finetune", unfreeze_epoch=5,
    freeze_bn_always=True,
    epochs=EPOCHS, patience=PATIENCE, monitor=MONITOR,
    grad_clip=GRAD_CLIP, use_amp=False,
    checkpoint_path=None, device=device,
    trial=None,
    lr=LR,
):
    '''Treina e valida por época, logando tudo no TensorBoard.

    Args:
        criterion: função de perda (vindo de LOSSES).
        run_name: nome do experimento (usado no log_dir e no checkpoint).
        backbone_name, loss_name: usados nos hparams do TensorBoard.
        freeze_mode: "full" | "frozen" | "finetune".
        unfreeze_epoch: época de descongelamento (só vale para "finetune").
        monitor: métrica de checkpoint/early-stopping ("ccc" | "mae" | "loss").
        use_amp: ativa mixed precision (mais rápido; cheque o ccc_loss em fp16).
    Returns:
        dict com as melhores métricas e o caminho do checkpoint.
    '''
    model.to(device)
    torch.cuda.empty_cache()
    if checkpoint_path is None:
        checkpoint_path = os.path.join(CKPT_DIR, f"{run_name}.pth")
    last_path = os.path.join(CKPT_DIR, f"{run_name}_last.pth")

    mode = MONITOR_MODE[monitor]
    best_score = -np.inf if mode == "max" else np.inf
    best_metrics, best_true, best_pred = {}, None, None
    epochs_no_improve = 0

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(RUNS_DIR, f"{run_name}_{stamp}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):
        # estado de congelamento desta época
        if freeze_mode == "full":
            backbone_frozen = False
        elif freeze_mode == "frozen":
            backbone_frozen = True
        else:  # finetune
            backbone_frozen = epoch < unfreeze_epoch
            if epoch == unfreeze_epoch:
                print(f"[época {epoch+1}] descongelando a ResNet (fine-tuning completo)")

        _apply_freeze(model, backbone_frozen)

        # ----- treino -----
        model.train()
        if freeze_bn_always or backbone_frozen:
            _set_backbone_eval(model)

        train_loss = 0.0
        train_bce = 0.0
        train_rank = 0.0

        train_true, train_pred = [], []
        
        for (low, high) in tqdm(train_loader, desc=f"época {epoch+1}/{epochs} [treino]", leave=False):
            low_video, _, low_mel, low_targets = low
            high_video, _, high_mel, high_targets = high

            low_video = low_video.to(device, non_blocking=True)
            high_video = high_video.to(device, non_blocking=True)

            low_mel = low_mel.to(device, non_blocking=True)
            high_mel = high_mel.to(device, non_blocking=True)

            low_targets = low_targets.to(device).float().view(-1)
            high_targets = high_targets.to(device).float().view(-1)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda", enabled=use_amp):
                low_outputs = model(low_video, low_mel).reshape(-1)
                high_outputs = model(high_video, high_mel).reshape(-1)

                loss, bce_loss, rank_loss = criterion(
                    low_targets,
                    high_targets,
                    low_outputs,
                    high_outputs,
                    return_components=True,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                grad_clip,
            )

            scaler.step(optimizer)
            scaler.update()

            batch_size = low_video.size(0)
            train_loss += loss.item() * batch_size
            train_bce += bce_loss.item() * batch_size
            train_rank += rank_loss.item() * batch_size

            preds = torch.cat([torch.sigmoid(low_outputs), torch.sigmoid(high_outputs)])
            targets = torch.cat([ low_targets,  high_targets])

            train_true.extend(targets.detach().cpu().numpy())
            train_pred.extend(preds.detach().cpu().numpy())
        
        train_loss /= len(train_loader.dataset)
        train_bce /= len(train_loader.dataset)
        train_rank /= len(train_loader.dataset)

        train_true, train_pred = np.array(train_true), np.array(train_pred)
        train_mae = np.mean(np.abs(train_true - train_pred))
        train_ccc = calculate_ccc(train_true, train_pred)
        train_acc = binary_accuracy(train_true, train_pred)

        # ----- validação -----
        model.eval()

        val_loss = 0.0
        val_bce = 0.0
        val_rank = 0.0

        all_true, all_pred = [], []

        with torch.no_grad():

            for (low, high) in tqdm(
                val_loader,
                desc=f"época {epoch+1}/{epochs} [val]",
                leave=False,
            ):

                low_video, _, low_mel, low_targets = low
                high_video, _, high_mel, high_targets = high

                low_video = low_video.to(device, non_blocking=True)
                high_video = high_video.to(device, non_blocking=True)

                low_mel = low_mel.to(device, non_blocking=True)
                high_mel = high_mel.to(device, non_blocking=True)

                low_targets = low_targets.to(device).float().view(-1)
                high_targets = high_targets.to(device).float().view(-1)

                with torch.amp.autocast("cuda", enabled=use_amp):

                    low_outputs = model(low_video, low_mel).reshape(-1)
                    high_outputs = model(high_video, high_mel).reshape(-1)

                    loss, bce_loss, rank_loss = criterion(
                        low_targets,
                        high_targets,
                        low_outputs,
                        high_outputs,
                        return_components=True,
                    )

                val_loss += loss.item() * low_video.size(0)
                val_bce += bce_loss.item() * low_video.size(0)
                val_rank += rank_loss.item() * low_video.size(0)

                preds = torch.cat([
                    torch.sigmoid(low_outputs),
                    torch.sigmoid(high_outputs),
                ])

                targets = torch.cat([
                    low_targets,
                    high_targets,
                ])

                all_true.extend(targets.detach().cpu().numpy())
                all_pred.extend(preds.detach().cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_bce /= len(val_loader.dataset)
        val_rank /= len(val_loader.dataset)

        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        
        pred_std = float(np.std(all_pred))
        target_std = float(np.std(all_true))
        val_mae = np.mean(np.abs(all_true - all_pred))
        val_ccc = calculate_ccc(all_true, all_pred)
        val_acc = binary_accuracy(all_true, all_pred)

        scheduler.step(val_loss)

        if trial is not None:
            trial.report(val_rank, epoch)
            if trial.should_prune():
                writer.close()
                raise optuna.TrialPruned()

        # ----- tensorboard -----
        writer.add_scalar("Loss/train_total", train_loss, epoch)
        writer.add_scalar("Loss/train_bce", train_bce, epoch)
        writer.add_scalar("Loss/train_rank", train_rank, epoch)

        writer.add_scalar("Loss/val_total", val_loss, epoch)
        writer.add_scalar("Loss/val_bce", val_bce, epoch)
        writer.add_scalar("Loss/val_rank", val_rank, epoch)

        writer.add_scalar("Metrics/MAE_train", train_mae, epoch)
        writer.add_scalar("Metrics/MAE_val", val_mae, epoch)
        writer.add_scalar("Metrics/CCC_train", train_ccc, epoch)
        writer.add_scalar("Metrics/CCC_val", val_ccc, epoch)
        writer.add_scalar("Metrics/Acc_train", train_acc, epoch)
        writer.add_scalar("Metrics/Acc_val", val_acc, epoch)
        writer.add_scalar("Diagnostics/pred_std", pred_std, epoch)
        writer.add_scalar("Diagnostics/target_std", target_std, epoch)

        writer.add_histogram("Val/predictions", all_pred, epoch)

        for gi, pg in enumerate(optimizer.param_groups):
            writer.add_scalar(f"Train/LR_group{gi}", pg["lr"], epoch)

        print(
            f"época [{epoch+1}/{epochs}]"
            f" | train {train_loss:.4f}"
            f" (bce {train_bce:.4f}, rank {train_rank:.4f})"
            f" | val {val_loss:.4f}"
            f" (bce {val_bce:.4f}, rank {val_rank:.4f})"
            f" | LR {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ----- checkpoint (best + last) + early stopping -----
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "run_name": run_name}, last_path)

        current = {"ccc": val_ccc, "mae": val_mae, "loss": val_loss}[monitor]
        if _is_better(current, best_score, mode):
            best_score = current
            best_metrics = {
                "val_loss": val_loss,
                "val_bce": val_bce,
                "val_rank": val_rank,
                "val_ccc": val_ccc,
                "val_acc": val_acc,
                "epoch": epoch,
            }

            best_true, best_pred = all_true, all_pred
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_metrics": best_metrics,
                "run_name": run_name,
            }, checkpoint_path)
            print(f"  novo melhor ({monitor}={best_score:.4f}) salvo em {checkpoint_path}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"early stopping (sem melhora por {patience} épocas)")
                break

    # scatter do melhor epoch — diagnóstico de variance collapse
    if best_pred is not None:
        _log_pred_scatter(writer, best_true, best_pred, "Val/pred_vs_target", best_metrics["epoch"])

    # hparams para comparar runs na aba HPARAMS (run_name="." liga aos scalars)
    writer.add_hparams(
        {"backbone": backbone_name, "loss": loss_name, "freeze_mode": freeze_mode,
         "freeze_bn_always": freeze_bn_always,
         "lr": lr, "hidden_size": MODEL_DEFAULTS["hidden_size"],
         "num_layers": MODEL_DEFAULTS["num_layers"], "frame_step": MODEL_DEFAULTS["frame_step"]},
        {
            "best/val_loss": best_metrics.get("val_loss", 0.0),
            "best/val_bce": best_metrics.get("val_bce", 0.0),
            "best/val_rank": best_metrics.get("val_rank", 0.0),
            "best/val_ccc": best_metrics.get("val_ccc", 0.0),
            "best/val_acc": best_metrics.get("val_acc", 0.0),
        },
        run_name=".",
    )
    writer.close()
    print(f"concluído. melhor {monitor}: {best_score:.4f}")
    return {"checkpoint": checkpoint_path, "best_metrics": best_metrics}

In [17]:
'''def run_experiment(
    audiomae,
    alpha=1.0,
    margin_scale=1.0,
    backbone_name="resnet18",
    freeze_mode="finetune",
    unfreeze_epoch=5,
    lr=LR,
    lr_backbone=None,
    weight_decay=WEIGHT_DECAY,
    model_kwargs=None,
    criterion=None,
    epochs=EPOCHS,
    patience=PATIENCE,
    use_amp=False,
    use_fusion=True,
    always_bn=False,
    train_loader=None,
    val_loader=None,
    trial=None,
):
    """Roda um experimento completo e retorna o resultado."""

    model_kwargs = {**MODEL_DEFAULTS, **(model_kwargs or {})}

    run_name = f"{backbone_name}__{freeze_mode}__fusion{use_fusion}__bn{always_bn}__trial{trial.number if trial else 'manual'}"

    print(f"\n=== {run_name} ===")

    model = ResNetLSTM_MultiModal(
        audiomae,
        backbone_name=backbone_name,
        use_fusion=use_fusion,
        **model_kwargs,
    ).to(device)

    if lr_backbone is None:

        optimizer = AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )

    else:

        optimizer = AdamW(
            [
                {"params": model.cnn.parameters(), "lr": lr_backbone},
                {"params": model.lstm.parameters(), "lr": lr},
                {"params": model.head.parameters(), "lr": lr},
            ],
            weight_decay=weight_decay,
        )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=3,
        factor=0.5,
    )

    if criterion is None:
        criterion = CombinedLoss(
            alpha=alpha,
            margin_scale=margin_scale,
        )

    return train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        run_name=run_name,
        optimizer=optimizer,
        scheduler=scheduler,
        backbone_name=backbone_name,
        loss_name=criterion.__class__.__name__,
        freeze_mode=freeze_mode,
        unfreeze_epoch=unfreeze_epoch,
        epochs=epochs,
        patience=patience,
        use_amp=use_amp,
        freeze_bn_always=always_bn,
        lr=lr,
        trial=trial,
    )'''

'def run_experiment(\n    audiomae,\n    alpha=1.0,\n    margin_scale=1.0,\n    backbone_name="resnet18",\n    freeze_mode="finetune",\n    unfreeze_epoch=5,\n    lr=LR,\n    lr_backbone=None,\n    weight_decay=WEIGHT_DECAY,\n    model_kwargs=None,\n    criterion=None,\n    epochs=EPOCHS,\n    patience=PATIENCE,\n    use_amp=False,\n    use_fusion=True,\n    always_bn=False,\n    train_loader=None,\n    val_loader=None,\n    trial=None,\n):\n    """Roda um experimento completo e retorna o resultado."""\n\n    model_kwargs = {**MODEL_DEFAULTS, **(model_kwargs or {})}\n\n    run_name = f"{backbone_name}__{freeze_mode}__fusion{use_fusion}__bn{always_bn}__trial{trial.number if trial else \'manual\'}"\n\n    print(f"\n=== {run_name} ===")\n\n    model = ResNetLSTM_MultiModal(\n        audiomae,\n        backbone_name=backbone_name,\n        use_fusion=use_fusion,\n        **model_kwargs,\n    ).to(device)\n\n    if lr_backbone is None:\n\n        optimizer = AdamW(\n            model.parame

In [18]:
# nova função sugerida pelo claude

def run_experiment(
    audiomae,
    alpha=1.0,
    margin_scale=1.0,
    backbone_name="resnet18",
    freeze_mode="finetune",
    unfreeze_epoch=5,
    lr=LR,
    lr_backbone=None,
    weight_decay=WEIGHT_DECAY,
    model_kwargs=None,
    criterion=None,
    epochs=EPOCHS,
    patience=PATIENCE,
    use_amp=True,
    use_fusion=True,
    always_bn=False,
    train_loader=None,
    val_loader=None,
    trial=None,
):
    model_kwargs = {**MODEL_DEFAULTS, **(model_kwargs or {})}
    run_name = f"{backbone_name}__{freeze_mode}__fusion{use_fusion}__bn{always_bn}__trial{trial.number if trial else 'manual'}"
    print(f"\n=== {run_name} ===")

    model = ResNetLSTM_MultiModal(
        audiomae,
        backbone_name=backbone_name,
        use_fusion=use_fusion,
        **model_kwargs,
    ).to(device)

    if lr_backbone is None:
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = AdamW(
            [
                {"params": model.cnn.parameters(), "lr": lr_backbone},
                {"params": model.lstm.parameters(), "lr": lr},
                {"params": model.head.parameters(), "lr": lr},
            ],
            weight_decay=weight_decay,
        )

    scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

    if criterion is None:
        criterion = CombinedLoss(alpha=alpha, margin_scale=margin_scale)

    try:
        result = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            run_name=run_name,
            optimizer=optimizer,
            scheduler=scheduler,
            backbone_name=backbone_name,
            loss_name=criterion.__class__.__name__,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=epochs,
            patience=patience,
            use_amp=use_amp,
            freeze_bn_always=always_bn,
            lr=lr,
            trial=trial,
        )
    finally:
        # Garante limpeza mesmo se o trial lançar exceção
        del model, optimizer, scheduler, criterion
        torch.cuda.empty_cache()
        import gc; gc.collect()

    return result

In [19]:
# definindo o modelo para embedding dos mel spectrogramas
model_ae = AutoModel.from_pretrained("hance-ai/audiomae", trust_remote_code=True).to(device)

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_ae = AutoModel.from_pretrained("hance-ai/audiomae", trust_remote_code=True).to(device)
FRAC_EPOCH = 0.1 # fracção dos dados que serão usados por época no treino
GROUPS = True # estamos usando o agrupamento por janela
GROUPS_COLUMN_ID = 'window_id' # coluna do agrupamento

'''
loaders = {
    bs: (
        build_multimodal_dataloader(
            csv_path=TRAIN_PATH, pair=True, split="train", batch_size=bs, shuffle=True,
            num_workers=4, is_grayscale=True, pin_memory=True,
            groups=GROUPS, column_groups_id = GROUPS_COLUMN_ID,
            video_transform=train_video_transform, mel_transform=train_mel_transform, target_shape=TARGET_SHAPE,
            epoch_frac=FRAC_EPOCH,
        ),
        build_multimodal_dataloader(
            csv_path=VAL_PATH, pair=True, split="valid", batch_size=bs, shuffle=False,
            num_workers=4, is_grayscale=True, pin_memory=True, target_shape=TARGET_SHAPE,
            groups=GROUPS, column_groups_id = GROUPS_COLUMN_ID,
        ),
    )
    for bs in [1, 2]
}
'''

FRAME_STEP = 8   # lê 1 frame a cada 2 → metade da RAM por vídeo

loaders = {
    bs: (
        build_multimodal_dataloader(
            csv_path=TRAIN_PATH, pair=True, split="train", batch_size=bs, shuffle=True,
            num_workers=2,            # era 4 — menos workers = menos vídeos simultâneos
            is_grayscale=True,
            pin_memory=False,         # era True — com vídeos grandes isso esgota RAM fixada
            groups=GROUPS, column_groups_id=GROUPS_COLUMN_ID,
            video_transform=train_video_transform, mel_transform=train_mel_transform,
            target_shape=TARGET_SHAPE, epoch_frac=FRAC_EPOCH,
            frame_step=FRAME_STEP,    # novo
        ),
        build_multimodal_dataloader(
            csv_path=VAL_PATH, pair=True, split="valid", batch_size=bs, shuffle=False,
            num_workers=2,
            is_grayscale=True,
            pin_memory=False,
            target_shape=TARGET_SHAPE,
            groups=GROUPS, column_groups_id=GROUPS_COLUMN_ID,
            frame_step=FRAME_STEP,    # novo
        ),
    )
    for bs in [1, 2]
}

Dataset de Treino: 61674/64464 exemplos válidos.
6109 grupos encontrados.
Low: 53650
High: 8024

890 pares de grupos criados.

Dataset de Validação: 17111/19462 exemplos válidos.
1674 grupos encontrados.
Low: 14619
High: 2492

274 pares de grupos criados.

Dataset de Treino: 61674/64464 exemplos válidos.
6109 grupos encontrados.
Low: 53650
High: 8024

890 pares de grupos criados.

Dataset de Validação: 17111/19462 exemplos válidos.
1674 grupos encontrados.
Low: 14619
High: 2492

274 pares de grupos criados.



Corrigindo possíveis arquivos corrompidos.

In [ ]:
SEARCH_EPOCHS = 40

'''
def objective(trial):
    backbone = trial.suggest_categorical("backbone", ["resnet18", "resnet34", "resnet50"])
    freeze_mode = trial.suggest_categorical("freeze_mode", ["frozen", "finetune"])
    use_fusion = trial.suggest_categorical("use_fusion", [True, False])
    always_bn = trial.suggest_categorical("always_bn", [True, False])
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    alpha = trial.suggest_float("alpha", 0.0, 1.0)
    margin_scale = trial.suggest_float("margin_scale", 0.5, 2.0)

    bs = 1 if backbone == "resnet50" else 2
    train_loader, val_loader = loaders[bs]

    try:
        result = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            freeze_mode=freeze_mode,
            unfreeze_epoch=5,
            alpha=alpha,
            margin_scale=margin_scale,
            use_fusion=use_fusion,
            always_bn=always_bn,
            lr=lr,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=SEARCH_EPOCHS,
            trial=trial,
        )
    except optuna.TrialPruned:
        raise
    except Exception as e:
        print(f"ERRO no trial {trial.number}: {e}")
        return float("inf")

    return result["best_metrics"]["val_rank"]
'''
'''
def objective(trial):
    backbone = trial.suggest_categorical("backbone", ["resnet18", "resnet34", "resnet50"])
    freeze_mode = trial.suggest_categorical("freeze_mode", ["frozen", "finetune"])
    use_fusion = trial.suggest_categorical("use_fusion", [True, False])
    always_bn = trial.suggest_categorical("always_bn", [True, False])
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    alpha = trial.suggest_float("alpha", 0.0, 1.0)
    margin_scale = trial.suggest_float("margin_scale", 0.5, 2.0)

    bs = 1 if backbone == "resnet50" else 2
    train_loader, val_loader = loaders[bs]

    try:
        result = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            freeze_mode=freeze_mode,
            unfreeze_epoch=5,
            alpha=alpha,
            margin_scale=margin_scale,
            use_fusion=use_fusion,
            always_bn=always_bn,
            lr=lr,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=SEARCH_EPOCHS,
            trial=trial,
        )

    except optuna.TrialPruned:
        raise

    except Exception as e:
        print(f"ERRO no trial {trial.number}: {e}")
        return float("inf")

    finally:
        # Libera memória da GPU entre os trials
        torch.cuda.empty_cache()
        import gc
        gc.collect()

    return result["best_metrics"]["val_rank"]


study = optuna.create_study(
    direction="minimize",
    study_name="multimodal_search",
    storage="sqlite:///optuna_multimodal.db",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.HyperbandPruner(min_resource=5, max_resource=SEARCH_EPOCHS),
)

study.optimize(objective, n_trials=20)'''


# sugerido pelo claude para reduzir o uso de gpu

# Congela o AudioMAE uma vez, fora dos trials
model_ae.eval()
for p in model_ae.parameters():
    p.requires_grad = False

def objective(trial):
    backbone = trial.suggest_categorical("backbone", ["resnet18", "resnet34", "resnet50"])
    freeze_mode = trial.suggest_categorical("freeze_mode", ["frozen", "finetune"])
    use_fusion = trial.suggest_categorical("use_fusion", [True, False])
    always_bn = trial.suggest_categorical("always_bn", [True, False])
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    alpha = trial.suggest_float("alpha", 0.0, 1.0)
    margin_scale = trial.suggest_float("margin_scale", 0.5, 2.0)

    bs = 1 if backbone == "resnet50" else 2
    train_loader, val_loader = loaders[bs]

    try:
        result = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            freeze_mode=freeze_mode,
            unfreeze_epoch=5,
            alpha=alpha,
            margin_scale=margin_scale,
            use_fusion=use_fusion,
            always_bn=always_bn,
            lr=lr,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=SEARCH_EPOCHS,
            trial=trial,
        )
    except optuna.TrialPruned:
        raise
    except Exception as e:
        print(f"ERRO no trial {trial.number}: {e}")
        torch.cuda.empty_cache()
        import gc; gc.collect()
        return float("inf")

    return result["best_metrics"]["val_rank"]


study = optuna.create_study(
    direction="minimize",
    study_name="multimodal_search_v5_fixed",
    storage="sqlite:///optuna_multimodal.db",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.HyperbandPruner(min_resource=5, max_resource=SEARCH_EPOCHS),
)

study.optimize(objective, n_trials=20)

print("\n=== MELHOR ===")
print(study.best_params)
print(study.best_value)

[I 2026-06-30 10:24:35,699] A new study created in RDB with name: multimodal_search_v5_fixed



=== resnet34__frozen__fusionTrue__bnTrue__trial0 ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal_daug_more_games/resnet34__frozen__fusionTrue__bnTrue__trial0_20260630-102435


época [1/40] | train 0.6271 (bce 0.6004, rank 1.2947) | val 6.0320 (bce 5.7965, rank 11.4413) | LR 2.61e-04
  novo melhor (loss=6.0320) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet34__frozen__fusionTrue__bnTrue__trial0.pth


época [2/40] | train 0.5958 (bce 0.5724, rank 1.1379) | val 6.2250 (bce 6.0124, rank 10.3258) | LR 2.61e-04


época [3/40] | train 0.5700 (bce 0.5507, rank 0.9379) | val 6.0827 (bce 5.9166, rank 8.0701) | LR 2.61e-04


...

época [26/40] | train 0.5039 (bce 0.4108, rank 0.0977) | val 7.0571 (bce 6.2046, rank 0.8948) | LR 5.72e-05


época [27/40] | train 0.5925 (bce 0.4663, rank 0.1324) | val 7.0568 (bce 6.1888, rank 0.9111) | LR 5.72e-05


época [28/40] | train 0.6231 (bce 0.4788, rank 0.1515) | val 5.3078 (bce 4.4496, rank 0.9007) | LR 5.72e-05
  novo melhor (loss=5.3078) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet18__finetune__fusionTrue__bnTrue__trial76.pth


época [29/40] | train 0.6209 (bce 0.4815, rank 0.1463) | val 5.3129 (bce 4.3228, rank 1.0392) | LR 5.72e-05


época 30/40 [treino]:   0%|          | 0/45 [00:00<?, ?it/s]